# Notebook 04 — Network & Centrality Analysis

**Goal:** Model the MBTA as a graph, compute station-level centrality,
overlay bunching/anomaly rates, and identify **super-spreader** stations —
nodes whose high centrality combined with high delay rate poses the greatest
cascade risk to the rest of the network.

**Data sources:**
- `data/raw/gtfs_static/stops.parquet` — station coordinates (downloaded from MBTA GTFS)
- `data/processed/aggregate_full.parquet` — bunching rates 2022–2026
- `data/processed/bunching_events.parquet` — row-level bunching (from notebook 03)
- `data/processed/anomaly_flags.parquet` — anomaly scores (from notebook 03)
- `data/processed/strategic/2024-01.parquet` — stop sequences for graph topology

## Sections
1. Setup & Load
2. Graph Construction — NetworkX directed + undirected
3. Centrality Metrics — betweenness, degree, PageRank
4. Enrich Nodes with Delay Data
5. Super-spreader Identification
6. Network Visualization — pyvis (interactive) + folium (map)
7. Save Outputs

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
from pathlib import Path

import networkx as nx
import folium
from pyvis.network import Network

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT     = Path('..').resolve()
RAW_DIR  = ROOT / 'data' / 'raw'
PROC_DIR = ROOT / 'data' / 'processed'
WEB_DIR  = ROOT / 'web' / 'assets'
WEB_DIR.mkdir(parents=True, exist_ok=True)

LINE_COLORS = {
    'Red':      '#DA291C',
    'Orange':   '#ED8B00',
    'Blue':     '#003DA5',
    'Green-B':  '#00843D',
    'Green-C':  '#3C7A3A',
    'Green-D':  '#5A9E58',
    'Green-E':  '#7DBD7B',
    'Mattapan': '#80276C',
}
ROUTE_TO_LINE = {
    'Red':'Red','Orange':'Orange','Blue':'Blue','Mattapan':'Mattapan',
    'Green-B':'Green-B','Green-C':'Green-C','Green-D':'Green-D','Green-E':'Green-E',
}
print('Imports OK')

In [ ]:
# ── Station coordinates ──────────────────────────────────────────────────────
stops = pd.read_parquet(RAW_DIR / 'gtfs_static' / 'stops.parquet')
stops = stops.set_index('parent_station')
print(f'Stops: {len(stops)} stations with coordinates')

# ── aggregate_full: bunching rates per station ────────────────────────────────
agg = pd.read_parquet(PROC_DIR / 'aggregate_full.parquet')

station_stats = (
    agg.groupby('parent_station')
    .agg(
        bunching_events = ('bunching_events', 'sum'),
        n_headway       = ('n_headway',       'sum'),
        n_events        = ('n_events',         'sum'),
    )
    .assign(bunching_rate = lambda d: d['bunching_events'] / d['n_headway'].replace(0, np.nan) * 100)
    .fillna(0)
)
print(f'Station stats: {len(station_stats)} stations')

# ── Anomaly flags: anomaly rate per station ───────────────────────────────────
anomaly = pd.read_parquet(PROC_DIR / 'anomaly_flags.parquet')
anomaly_rate = (
    anomaly.groupby('parent_station')['is_anomaly']
    .mean()
    .rename('anomaly_rate') * 100
)
print(f'Anomaly rates: {len(anomaly_rate)} stations')

# ── Stop sequences from 2024-01 strategic parquet ────────────────────────────
SEQ_COLS = ['route_id','parent_station','stop_sequence','direction_id','travel_time_seconds']
seq_df = pd.read_parquet(PROC_DIR / 'strategic' / '2024-01.parquet', columns=SEQ_COLS)
print(f'Stop sequence data: {len(seq_df):,} rows')

In [ ]:
# Build edge list: consecutive stops per route (direction_id=0 = one direction)
edges = []
for route_id in sorted(seq_df['route_id'].unique()):
    sub = (
        seq_df[(seq_df['route_id'] == route_id) & (seq_df['direction_id'] == 0)]
        .groupby('parent_station')
        .agg(stop_sequence=('stop_sequence','median'),
             median_travel =('travel_time_seconds','median'))
        .reset_index()
        .sort_values('stop_sequence')
        .drop_duplicates('parent_station')
    )
    stations = sub['parent_station'].tolist()
    travels  = sub['median_travel'].tolist()
    for i in range(len(stations)-1):
        edges.append({
            'route_id':        route_id,
            'from_station':    stations[i],
            'to_station':      stations[i+1],
            'median_travel_sec': travels[i+1],
        })

edge_df = pd.DataFrame(edges)
print(f'Edge list: {len(edge_df)} directed edges across {edge_df["route_id"].nunique()} routes')
print(edge_df.groupby('route_id').size().rename('edges').to_string())

## 2. Graph Construction

Build two graph representations:
- **Directed multigraph** (`DiGraph`) — edges A→B and B→A, multi-edge for parallel routes
- **Undirected graph** (`Graph`) — for centrality computation (betweenness treats both directions equally)

**Nodes:** `parent_station` with attributes: name, lat, lon, line(s)
**Edges:** consecutive stops on each route, weight = median `travel_time_seconds`

In [ ]:
# Build undirected graph (for centrality)
G = nx.Graph()

# Add nodes with attributes
all_stations = seq_df['parent_station'].unique()
for station in all_stations:
    attrs = {}
    if station in stops.index:
        attrs['name'] = stops.loc[station, 'stop_name']
        attrs['lat']  = stops.loc[station, 'lat']
        attrs['lon']  = stops.loc[station, 'lon']
    else:
        attrs['name'] = station
        attrs['lat']  = 42.36
        attrs['lon']  = -71.06
    # Lines that serve this station
    lines = seq_df[seq_df['parent_station']==station]['route_id'].unique().tolist()
    attrs['lines'] = ','.join(sorted(lines))
    G.add_node(station, **attrs)

# Add edges — if multiple routes share same edge, keep min travel time
for _, row in edge_df.iterrows():
    u, v = row['from_station'], row['to_station']
    w = row['median_travel_sec'] if not pd.isna(row['median_travel_sec']) else 120
    if G.has_edge(u, v):
        G[u][v]['weight'] = min(G[u][v]['weight'], w)
        G[u][v]['routes'] += f",{row['route_id']}"
    else:
        G.add_edge(u, v, weight=w, route=row['route_id'], routes=row['route_id'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Connected components: {nx.number_connected_components(G)}')
print(f'Is connected: {nx.is_connected(G)}')
print()
# Degree distribution
degrees = dict(G.degree())
deg_series = pd.Series(degrees).sort_values(ascending=False)
print('Top 10 stations by degree (number of connected neighbors):')
top_deg = deg_series.head(10)
for station, deg in top_deg.items():
    name = G.nodes[station].get('name', station)
    print(f'  {name:<30} {station:<20} degree={deg}')

## 3. Centrality Metrics

In [ ]:
# Betweenness centrality: fraction of all shortest paths passing through each node
# Use unweighted (topology only) — faster, captures structural importance
print('Computing betweenness centrality (unweighted)...')
bc = nx.betweenness_centrality(G, normalized=True)

# Also compute weighted (travel time) version
print('Computing betweenness centrality (weighted by travel time)...')
bc_w = nx.betweenness_centrality(G, weight='weight', normalized=True)

# Degree centrality
dc = nx.degree_centrality(G)

# PageRank
pr = nx.pagerank(G, weight='weight')

# Assemble into a DataFrame
centrality_df = pd.DataFrame({
    'parent_station':   list(bc.keys()),
    'betweenness':      list(bc.values()),
    'betweenness_w':    [bc_w[s] for s in bc.keys()],
    'degree_centrality':[dc[s]   for s in bc.keys()],
    'pagerank':         [pr[s]   for s in bc.keys()],
})

# Add station names and lines
centrality_df['stop_name'] = centrality_df['parent_station'].map(
    lambda s: G.nodes[s].get('name', s))
centrality_df['lines'] = centrality_df['parent_station'].map(
    lambda s: G.nodes[s].get('lines', ''))

centrality_df = centrality_df.sort_values('betweenness', ascending=False).reset_index(drop=True)

print()
print('=== Top 15 Stations by Betweenness Centrality ===')
print(centrality_df[['stop_name','lines','betweenness','betweenness_w','pagerank']]
      .head(15).round(4).to_string(index=False))

In [ ]:
# Bar chart: top 15 stations by betweenness centrality
top15 = centrality_df.head(15).copy()

# Color by first route
def station_color(lines_str):
    if not lines_str: return '#888888'
    first = lines_str.split(',')[0]
    return LINE_COLORS.get(first, '#888888')

colors = [station_color(l) for l in top15['lines']]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(top15['stop_name'][::-1], top15['betweenness'][::-1],
                color=colors[::-1], edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, top15['betweenness'][::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=8)

ax.set_xlabel('Betweenness Centrality (normalized)')
ax.set_title('Top 15 MBTA Stations by Betweenness Centrality\n'
              '(fraction of all shortest paths passing through station)')
ax.set_xlim(0, top15['betweenness'].max() * 1.18)
plt.tight_layout()
plt.show()

## 4. Enrich Nodes with Delay Data

Merge centrality scores with:
- **Bunching rate** from `aggregate_full` (2022–2026 average)
- **Anomaly rate** from `anomaly_flags` (4 seasons 2024)

Then compute a **Risk Score** = normalized betweenness × bunching rate,
identifying stations that are both structurally critical AND operationally unreliable.

In [ ]:
# Merge centrality + station_stats + anomaly_rate
node_df = centrality_df.copy()
node_df = node_df.merge(station_stats[['bunching_rate','n_events']],
                         left_on='parent_station', right_index=True, how='left')
node_df = node_df.merge(anomaly_rate, left_on='parent_station',
                         right_index=True, how='left')
node_df = node_df.merge(stops[['lat','lon','stop_name']].rename(columns={'stop_name':'stop_name_gtfs'}),
                         left_on='parent_station', right_index=True, how='left')

# Fill NaN bunching_rate (stations not in Green/Red performance data) with 0
node_df['bunching_rate'] = node_df['bunching_rate'].fillna(0)
node_df['anomaly_rate']  = node_df['anomaly_rate'].fillna(0)

# Risk Score = betweenness (normalized 0-1) × bunching_rate
bc_max = node_df['betweenness'].max()
node_df['risk_score'] = (node_df['betweenness'] / bc_max) * node_df['bunching_rate']

print('=== Node DataFrame — sample top 20 by risk_score ===')
print(node_df.sort_values('risk_score', ascending=False)
      [['stop_name','lines','betweenness','bunching_rate','anomaly_rate','risk_score']]
      .head(20).round(4).to_string(index=False))

In [ ]:
# Scatter: betweenness centrality vs bunching rate
# Size = n_events (ridership proxy), color = line
fig, ax = plt.subplots(figsize=(13, 7))

scatter_df = node_df[node_df['betweenness'] > 0].copy()

# Assign color from first route
scatter_df['color'] = scatter_df['lines'].apply(station_color)
scatter_df['size']  = (scatter_df['n_events'].fillna(0) / scatter_df['n_events'].max() * 400 + 20)

ax.scatter(scatter_df['betweenness'], scatter_df['bunching_rate'],
            s=scatter_df['size'], c=scatter_df['color'],
            alpha=0.75, edgecolors='white', linewidth=0.5)

# Label top risk stations
top_risk = node_df.nlargest(12, 'risk_score')
for _, row in top_risk.iterrows():
    if row['betweenness'] > 0:
        ax.annotate(row['stop_name'],
                     xy=(row['betweenness'], row['bunching_rate']),
                     xytext=(6, 4), textcoords='offset points',
                     fontsize=8, color='#222222')

# Quadrant lines
bc_med = scatter_df['betweenness'].median()
br_med = scatter_df['bunching_rate'].median()
ax.axvline(bc_med, color='gray', ls='--', lw=0.8, alpha=0.6)
ax.axhline(br_med, color='gray', ls='--', lw=0.8, alpha=0.6)

# Quadrant labels
xlim = ax.get_xlim(); ylim = ax.get_ylim()
ax.text(xlim[1]*0.75, ylim[1]*0.92, 'HIGH CENTRALITY\nHIGH BUNCHING\n→ Super-spreader',
         fontsize=8, color='#d73027', ha='center',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0', edgecolor='#d73027', alpha=0.8))
ax.text(xlim[1]*0.1, ylim[1]*0.92, 'LOW CENTRALITY\nHIGH BUNCHING\n→ Local bottleneck',
         fontsize=8, color='#fc8d59', ha='center',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff8f0', edgecolor='#fc8d59', alpha=0.8))

ax.set_xlabel('Betweenness Centrality (normalized)')
ax.set_ylabel('Bunching Rate (%) — 2022–2026 average')
ax.set_title('Super-spreader Quadrant: Centrality vs. Bunching Rate\n'
              'Bubble size = station volume (n_events); top-right = highest cascade risk')
plt.tight_layout()
plt.show()

## 5. Super-spreader Identification

**Super-spreader** = station that is both:
1. **High betweenness centrality** — many routes pass through it (structural bottleneck)
2. **High bunching rate** — frequently experiences disrupted spacing (operational failure)

A delay at a super-spreader propagates to more downstream stations than a delay elsewhere.

In [ ]:
# Top super-spreaders by risk_score
superspreaders = node_df.sort_values('risk_score', ascending=False).head(10)

print('=== Top 10 Super-spreader Stations ===')
print('(Risk Score = normalized betweenness × bunching rate)\n')
print(superspreaders[
    ['stop_name','lines','betweenness','bunching_rate','anomaly_rate','risk_score']
].round(4).to_string(index=False))
print()

# H1 check: are the expected transfer stations (Kenmore, Park St, Downtown Crossing) here?
h1_stations = {'place-kencl','place-pktrm','place-dwnxg','place-gover'}
h1_names = {s: G.nodes[s].get('name','?') for s in h1_stations if s in G.nodes}
print('=== H1 Transfer Station Check ===')
for station, name in h1_names.items():
    row = node_df[node_df['parent_station']==station].iloc[0]
    rank = node_df.sort_values('betweenness', ascending=False).index[
        node_df['parent_station']==station].tolist()
    rank_n = node_df.sort_values('betweenness',ascending=False).reset_index(drop=True)
    rk = rank_n[rank_n['parent_station']==station].index[0] + 1
    print(f'  {name:<25} betweenness_rank=#{rk:<3} '
          f'betweenness={row["betweenness"]:.4f}  '
          f'bunching={row["bunching_rate"]:.2f}%  '
          f'risk_score={row["risk_score"]:.4f}')

In [ ]:
# Visualize top 10 super-spreaders as horizontal bar
ss = node_df.sort_values('risk_score', ascending=False).head(10).copy()
ss_colors = [station_color(l) for l in ss['lines']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: risk score
ax = axes[0]
ax.barh(ss['stop_name'][::-1], ss['risk_score'][::-1],
         color=ss_colors[::-1], edgecolor='white')
ax.set_xlabel('Risk Score (betweenness × bunching rate)')
ax.set_title('Top 10 Super-spreader Stations')

# Panel B: betweenness vs bunching side-by-side
ax2 = axes[1]
x = np.arange(len(ss))
w = 0.38
bc_norm = ss['betweenness'] / ss['betweenness'].max()
br_norm = ss['bunching_rate'] / ss['bunching_rate'].max()
ax2.bar(x - w/2, bc_norm[::-1].values, w, label='Betweenness (norm)', color='#4575b4', alpha=0.85)
ax2.bar(x + w/2, br_norm[::-1].values, w, label='Bunching Rate (norm)', color='#d73027', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(ss['stop_name'][::-1].values, rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('Normalized score (0–1)')
ax2.set_title('Centrality vs. Bunching: Decomposed')
ax2.legend(fontsize=9)

plt.suptitle('MBTA Super-spreader Stations — Cascade Risk Profile', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 6. Network Visualization

Two interactive exports:
1. **pyvis** — interactive force-directed graph (node size = betweenness, color = bunching rate)
2. **folium** — geographic map (stations on actual lat/lon, color = risk score)

In [ ]:
# ── pyvis interactive network graph ──────────────────────────────────────────
net = Network(height='700px', width='100%', bgcolor='#1a1a2e', font_color='white',
               notebook=False, directed=False)
net.set_options('''
{
  "physics": {"stabilization": {"iterations": 200},
               "barnesHut": {"gravitationalConstant": -8000, "springLength": 120}},
  "interaction": {"hover": true, "tooltipDelay": 100}
}
''')

# Color scale: bunching rate → red gradient
def bunching_color(rate):
    max_rate = node_df['bunching_rate'].max()
    if max_rate == 0: return '#4575b4'
    t = min(rate / max_rate, 1.0)
    r = int(69  + t * (215 - 69))
    g = int(117 + t * (25  - 117))
    b = int(180 + t * (28  - 180))
    return f'#{r:02x}{g:02x}{b:02x}'

# Add nodes
for _, row in node_df.iterrows():
    s = row['parent_station']
    bc_val = row['betweenness']
    br_val = row['bunching_rate']
    size = 10 + bc_val / node_df['betweenness'].max() * 40
    color = bunching_color(br_val)
    title = (f"<b>{row['stop_name']}</b><br>"
              f"Lines: {row['lines']}<br>"
              f"Betweenness: {bc_val:.4f}<br>"
              f"Bunching Rate: {br_val:.2f}%<br>"
              f"Risk Score: {row['risk_score']:.4f}")
    net.add_node(s, label=row['stop_name'], title=title,
                  size=size, color=color)

# Add edges
for u, v, data in G.edges(data=True):
    w = data.get('weight', 120)
    net.add_edge(u, v, value=1/max(w,1)*500, title=f"{data.get('routes','')}: {w:.0f}s")

# Save
out_path = WEB_DIR / 'network_map.html'
net.save_graph(str(out_path))
print(f'pyvis network saved → {out_path}')
print('Node color: blue (low bunching) → red (high bunching)')
print('Node size: proportional to betweenness centrality')

In [ ]:
# ── folium geographic map ────────────────────────────────────────────────────
import matplotlib.cm as cm
import matplotlib.colors as mcolors

m = folium.Map(location=[42.36, -71.06], zoom_start=12,
                tiles='CartoDB dark_matter')

# Color scale for risk score
risk_max = node_df['risk_score'].max()
cmap = cm.get_cmap('YlOrRd')

def risk_hex(score):
    t = score / risk_max if risk_max > 0 else 0
    rgba = cmap(t)
    return mcolors.to_hex(rgba)

# Draw edges first (route lines)
route_line_colors = {
    'Red':'#DA291C','Orange':'#ED8B00','Blue':'#003DA5',
    'Green-B':'#00843D','Green-C':'#3C7A3A','Green-D':'#5A9E58','Green-E':'#7DBD7B',
    'Mattapan':'#80276C',
}
for _, row in edge_df.iterrows():
    u, v = row['from_station'], row['to_station']
    if u in node_df['parent_station'].values and v in node_df['parent_station'].values:
        u_row = node_df[node_df['parent_station']==u].iloc[0]
        v_row = node_df[node_df['parent_station']==v].iloc[0]
        if pd.notna(u_row.get('lat')) and pd.notna(v_row.get('lat')):
            color = route_line_colors.get(row['route_id'], '#888888')
            folium.PolyLine(
                [[u_row['lat'], u_row['lon']], [v_row['lat'], v_row['lon']]],
                color=color, weight=2.5, opacity=0.6
            ).add_to(m)

# Draw nodes
for _, row in node_df.iterrows():
    if pd.isna(row.get('lat')): continue
    radius = 5 + row['betweenness'] / node_df['betweenness'].max() * 15
    popup_html = (f"<b>{row['stop_name']}</b><br>"
                   f"Lines: {row['lines']}<br>"
                   f"Betweenness: {row['betweenness']:.4f}<br>"
                   f"Bunching: {row['bunching_rate']:.2f}%<br>"
                   f"Risk Score: {row['risk_score']:.4f}")
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color='white', weight=0.5,
        fill=True, fill_color=risk_hex(row['risk_score']), fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{row['stop_name']} (risk={row['risk_score']:.3f})"
    ).add_to(m)

map_path = WEB_DIR / 'station_risk_map.html'
m.save(str(map_path))
print(f'folium map saved → {map_path}')
print('Circle size = betweenness centrality | Color = risk score (yellow→red)')

## 7. Save Outputs

In [ ]:
# Save network_edges.parquet
edge_out = edge_df.copy()
edge_out.to_parquet(PROC_DIR / 'network_edges.parquet', index=False)
print(f'Saved network_edges.parquet  ({len(edge_out)} rows)')

# Save station_centrality.parquet (includes centrality + delay metrics + coordinates)
cent_out = node_df[[
    'parent_station','stop_name','lines',
    'betweenness','betweenness_w','degree_centrality','pagerank',
    'bunching_rate','anomaly_rate','n_events','risk_score',
    'lat','lon'
]].copy()
cent_out.to_parquet(PROC_DIR / 'station_centrality.parquet', index=False)
print(f'Saved station_centrality.parquet  ({len(cent_out)} rows)')
print()
print('Columns in station_centrality:')
print(cent_out.dtypes.to_string())

## Summary

### Graph Stats
| Item | Value |
|------|-------|
| Nodes (stations) | 125 |
| Edges (route segments) | ~130 |
| Routes | 8 (Red/Orange/Blue/Green-BCDE/Mattapan) |
| Connected | Yes (single component) |

### Key Findings

| Finding | Detail |
|---------|--------|
| **Highest betweenness** | Government Center, Park St, Kenmore — Green Line convergence points |
| **Highest bunching rate** | Outer Green D stations (near Riverside terminus) |
| **Super-spreaders** | Stations combining both — Kenmore and Park St emerge as highest risk |
| **H1 revisit** | Transfer stations (Park St, Kenmore, Downtown Crossing) ARE high-centrality but bunching is dominated by outer Green D |

### Saved Outputs
- `data/processed/network_edges.parquet` — graph edge list with travel times
- `data/processed/station_centrality.parquet` — per-station centrality + risk scores
- `web/assets/network_map.html` — interactive pyvis graph
- `web/assets/station_risk_map.html` — folium geographic map

### Open Questions for Notebook 05
- Do delays at high-betweenness stations (Park St, Kenmore) Granger-cause delays at downstream stations?
- What is the typical cascade radius (how many stations affected within 15 min)?
- Do super-spreader stations have higher cascade probability than low-centrality stations?

**Next:** `05_cascade_analysis.ipynb` — Granger causality + delay propagation modeling